# A par swap coupon from discounted cashflows

**Question.** What fixed coupon makes a five-year receive-fixed USD swap worth
zero, and does the native fair-rate solve agree with an annuity calculation?

Assume USD 1 million notional, origination on 24 February 2026, Actual/360
accrual, unadjusted dates, semiannual fixed payments and quarterly floating
payments. A single SOFR curve projects and discounts; no credit or funding
spread is applied. [Setup](../../README.md#install-and-run).

For a spot-starting single-curve swap, the projected floating leg telescopes:
$$V(K)=N\left[K A-(1-P(0,T))\right],\qquad
A=\sum_i\tau_i P(0,T_i),\qquad K^*=\frac{1-P(0,T)}{A}.$$
The annuity $A$ has units of years; $K$ is an annual decimal rate. Different
projection and discount curves would require the explicit floating-leg sum.


In [1]:
import pandas as pd
import quantsupport as qs
from market_data import load_market

market = load_market()
ref = market.quotes.reference_date


## A swap: value the contract and solve its par coupon

The fixed rate is the term we vary. Everything else stays identical. Explicit
trade dates matter for forward-starting products: the default trade date on
many specifications is their start date, while we want valuation as of `ref`.

In [2]:
def receive_fixed_swap(fixed_rate):
    instrument = qs.Swap(
        identifier="USD-SWAP-5Y", start_date=ref, maturity_date=ref + "5Y",
        notional=1_000_000, fixed_rate=fixed_rate,
        currency=qs.Currency.USD, market_index=qs.MarketIndex.SOFR,
        day_counter=qs.DayCounter.Actual360,
        fixed_leg_frequency=qs.Frequency.Semiannual,
        floating_leg_frequency=qs.Frequency.Quarterly,
    )
    return qs.Trade(instrument, ref, qs.Side.LongReceive)


context = qs.PricingContext(
    quotes=market.quotes, curves=market.curves, csa_policy=market.csa_policy,
)
with context:
    curve = context.curve(qs.MarketIndex.SOFR)
    payment_dates = [ref + qs.Period(6 * period, qs.TimeUnit.Months) for period in range(11)]
    fixed_schedule = pd.DataFrame([
        {
            "payment_date": str(end),
            "accrual": qs.DayCounter.Actual360.year_fraction(start, end),
            "discount_factor": curve.discount_factor(end),
        }
        for start, end in zip(payment_dates, payment_dates[1:])
    ])
    annuity = (fixed_schedule["accrual"] * fixed_schedule["discount_factor"]).sum()
    floating_pv_per_notional = 1 - curve.discount_factor(ref + "5Y")
    independently_solved_rate = floating_pv_per_notional / annuity
    swap_result = context.evaluate(receive_fixed_swap(0.04), [qs.Request.Value, qs.Request.FairRate])
    par_result = context.evaluate(receive_fixed_swap(swap_result.fair_rate), [qs.Request.Value])

pd.Series({
    "fixed_annuity_years": annuity,
    "native_par_rate": swap_result.fair_rate,
    "independent_par_rate": independently_solved_rate,
    "original_pv_usd": swap_result.price,
    "independent_original_pv_usd": 1_000_000 * (0.04 * annuity - floating_pv_per_notional),
    "pv_at_par_usd": par_result.price,
})


fixed_annuity_years            4.524195e+00
native_par_rate                4.036800e-02
independent_par_rate           4.036800e-02
original_pv_usd               -1.664892e+03
independent_original_pv_usd   -1.664892e+03
pv_at_par_usd                 -2.728484e-11
dtype: float64

In [3]:
import math

assert math.isclose(swap_result.fair_rate, independently_solved_rate, abs_tol=1e-9)
assert math.isclose(
    swap_result.price, 1_000_000 * annuity * (0.04 - independently_solved_rate), abs_tol=1e-5,
)
assert abs(par_result.price) < 1e-5
print(f"Par coupon: {swap_result.fair_rate:.6%}; PV at par: USD {par_result.price:.3e}.")
print(f"One coupon bp changes fixed-leg PV by USD {1_000_000 * annuity * 1e-4:.2f}.")
fixed_schedule


Par coupon: 4.036800%; PV at par: USD -2.728e-11.
One coupon bp changes fixed-leg PV by USD 452.42.


,payment_date,accrual,discount_factor
0,2026-08-24,0.502778,0.976544
1,2027-02-24,0.511111,0.953257
2,2027-08-24,0.502778,0.933651
3,2028-02-24,0.511111,0.914227
4,2028-08-24,0.505556,0.897578
5,2029-02-24,0.511111,0.881127
6,2029-08-24,0.502778,0.864875
7,2030-02-24,0.511111,0.848652
8,2030-08-24,0.502778,0.832992
9,2031-02-24,0.511111,0.817367


## Conclusion

The independent annuity calculation matches the native fair coupon and the
repriced swap is worth zero. A higher contractual fixed coupon increases this
receive-fixed position's value linearly **at a fixed market**. This coupon
exposure is not the same as market-quote DV01, which also changes discounting
and projected floating coupons.

Continue with [currency conversion](currency_conversion.ipynb) or
[portfolio quote risk](../../sensitivity/portfolio_risk.ipynb).
